In [ ]:
#| default_exp spec

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Build kernel specifications, bootstrap source, and execution results. Kernel startup lives in `kunda.kernel`.

In [ ]:
#| export
from __future__ import annotations
import importlib.util, os, subprocess, sys
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.all import L, first, ifnone
from jupyter_client.kernelspec import KernelSpec
from kunda.support import HOST_PY, clean_env, support_paths
from kunda.pythons import app_name

In [ ]:
#| export
def run_file_src(path, src=None, argv=(), cwd=None):
    "Code that runs a file in the kernel's *own* namespace, with script semantics."
    src = open(path, encoding='utf-8').read() if src is None else src
    setup = ('import os as _kd_os, sys as _kd_sys\n'
        f'_kd_code = compile({src!r}, {str(path)!r}, "exec")\n'
        '_kd_argv, _kd_cwd = _kd_sys.argv, _kd_os.getcwd()\n'
        f'_kd_sys.argv = [{str(path)!r}, *{list(argv)!r}]\n')
    if cwd: setup += f'_kd_os.chdir({str(cwd)!r})\n'
    return setup + ('try: exec(_kd_code)\n'
        'finally:\n'
        '    _kd_sys.argv = _kd_argv\n'
        '    _kd_os.chdir(_kd_cwd)\n'
        '    del _kd_os, _kd_sys, _kd_code, _kd_argv, _kd_cwd\n')

`run_file_src` runs a file in the kernel namespace. It sets `sys.argv` and an optional working directory for the run, then restores both. `src` can supply unsaved editor text. Tracebacks still use `path`.

In [ ]:
print(run_file_src('/proj/train.py', src='import sys\nprint(sys.argv)\n', argv=['--epochs', '3']))

import os as _kd_os, sys as _kd_sys
_kd_code = compile('import sys\nprint(sys.argv)\n', '/proj/train.py', "exec")
_kd_argv, _kd_cwd = _kd_sys.argv, _kd_os.getcwd()
_kd_sys.argv = ['/proj/train.py', *['--epochs', '3']]
try: exec(_kd_code)
finally:
    _kd_sys.argv = _kd_argv
    _kd_os.chdir(_kd_cwd)
    del _kd_os, _kd_sys, _kd_code, _kd_argv, _kd_cwd



In [ ]:
#| hide
tmp = TemporaryDirectory(); d = Path(tmp.name)
(d/'greet.py').write_text('import os, sys\ngreeting = f"hello {sys.argv[1]}"\nran_in = os.getcwd()\n')
argv0, cwd0, ns = list(sys.argv), os.getcwd(), {}
exec(run_file_src(d/'greet.py', argv=['world'], cwd=d), ns)
test_eq(ns['greeting'], 'hello world')
test_eq(Path(ns['ran_in']).resolve(), d.resolve())
assert not [k for k in ns if k.startswith('_kd_')], 'the setup leaves nothing behind'
test_fail(lambda: exec(run_file_src(d/'greet.py', src='raise ValueError("boom")\n'), {}), contains='boom')
test_eq(sys.argv, argv0)
test_eq(os.getcwd(), cwd0)

In [ ]:
#| export
BOOTSTRAP = '''
def _kd_bootstrap():
	import sys, importlib.util
	if tuple(sys.version_info[:2]) == tuple({version!r}):
		for _p in {support!r}:
			if _p and _p not in sys.path: sys.path.append(_p)
	try:
		if tuple(sys.version_info[:2]) != tuple({version!r}): raise ImportError('another Python')
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
		_spec = importlib.util.spec_from_file_location('dhrishti', {dhrishti_init!r}, submodule_search_locations=[{dhrishti_dir!r}])
		_pkg = importlib.util.module_from_spec(_spec)
		sys.modules['dhrishti'] = _pkg
		_spec.loader.exec_module(_pkg)
	except Exception:
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
	import dhrishti.serving as ls
	r = ls.serve_in_kernel(name={name!r}, port={port!r}, agent={agent!r}, token={token!r}, session_dir={sessions!r}, agent_session_dir={agent_sessions!r})
	return r
try: _kd_bootstrap()
finally: del _kd_bootstrap
'''

`BOOTSTRAP` starts Dhrishti inside the kernel. Host import paths are used only when the kernel has the same Python minor version. The host Dhrishti package is preferred, with the project package as fallback. Temporary bootstrap names are removed.

In [ ]:
args = dict(name='nb-1', port=8123, agent='restricted', token=True, support=['/host/site-packages'],
            sessions='_kunda_sessions', agent_sessions='_kunda_agent_sessions', version=(3, 12),
            dhrishti_init='/host/dhrishti/__init__.py', dhrishti_dir='/host/dhrishti')
src = BOOTSTRAP.format(**args)
print(src)


def _kd_bootstrap():
	import sys, importlib.util
	if tuple(sys.version_info[:2]) == tuple((3, 12)):
		for _p in ['/host/site-packages']:
			if _p and _p not in sys.path: sys.path.append(_p)
	try:
		if tuple(sys.version_info[:2]) != tuple((3, 12)): raise ImportError('another Python')
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
		_spec = importlib.util.spec_from_file_location('dhrishti', '/host/dhrishti/__init__.py', submodule_search_locations=['/host/dhrishti'])
		_pkg = importlib.util.module_from_spec(_spec)
		sys.modules['dhrishti'] = _pkg
		_spec.loader.exec_module(_pkg)
	except Exception:
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]: del sys.modules[_n]
	import dhrishti.serving as ls
	r = ls.serve_in_kernel(name='nb-1', port=8123, agent='restricted', token=True, session_dir='_kunda_sessions', agent_session_dir='_kunda_agent_sessions')
	return r
try: _kd_bootstrap()
finally: del _kd

In [ ]:
#| hide
assert compile(src, '<bootstrap>', 'exec')
test_eq(src.count('(3, 12)'), 2)
assert 'set_kernel_backend' not in src
assert 'get_ipython' not in src

In [ ]:
#| export
def bootstrap_src(name=None, port=8000, agent='restricted', token=True, sessions=None, agent_sessions=None):
    "The bootstrap cell source for a kernel that should host an inspector."
    sessions, agent_sessions = (ifnone(sessions, f'_{app_name()}_sessions'), ifnone(agent_sessions, f'_{app_name()}_agent_sessions'))
    support = support_paths()
    spec = importlib.util.find_spec('dhrishti')
    if spec is None or not spec.origin: raise RuntimeError('dhrishti is not installed')
    dhrishti_init = str(Path(spec.origin).resolve())
    return BOOTSTRAP.format(name=name, port=port, agent=agent, token=token, support=support, sessions=sessions, agent_sessions=agent_sessions,
        version=HOST_PY, dhrishti_init=dhrishti_init, dhrishti_dir=str(Path(dhrishti_init).parent))

`bootstrap_src` fills the bootstrap template. It finds Dhrishti without importing it. A missing Dhrishti installation raises `RuntimeError`.

In [ ]:
#| hide
if importlib.util.find_spec('dhrishti') is None: test_fail(bootstrap_src, contains='dhrishti is not installed')
else: assert compile(bootstrap_src(name='nb-1'), '<bootstrap>', 'exec')
from kunda.pythons import use_app
if importlib.util.find_spec('dhrishti') is not None:
    test_eq(bootstrap_src().count("session_dir='_kunda_sessions'"), 1)
    use_app('leela', 'LEELA_')
    test_eq(bootstrap_src().count("agent_session_dir='_leela_agent_sessions'"), 1)
    test_eq(bootstrap_src(sessions='_own').count("session_dir='_own'"), 1)   # a caller still wins
    use_app()

In [ ]:
#| export
def output_text(outs):
    "Flatten nbformat outputs to plain text: the terminal rendering, and the agent's view of a run."
    parts = L()
    for o in outs:
        t = o.get('output_type')
        if t == 'stream': parts.append(o.get('text', ''))
        elif t == 'error': parts.append('\n'.join(o.get('traceback') or [f"{o.get('ename')}: {o.get('evalue')}"]))
        elif t in ('execute_result', 'display_data'):
            d = o.get('data') or {}
            parts.append(d.get('text/markdown') or d.get('text/plain') or next((f'[{k}]' for k in d), ''))
    return ''.join(parts)

`output_text` converts nbformat outputs to terminal text. It includes streams, tracebacks, Markdown, and plain text. Other display data is represented by its MIME type.

In [ ]:
outs = [{'output_type': 'stream', 'name': 'stdout', 'text': 'fitting\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '0.94'}},
        {'output_type': 'display_data', 'data': {'image/png': 'iVBORw0KGgo'}}]
print(output_text(outs))

fitting
0.94[image/png]


In [ ]:
#| hide
test_eq(output_text([]), '')
test_eq(output_text([{'output_type': 'error', 'ename': 'ValueError', 'evalue': 'boom'}]), 'ValueError: boom')
test_eq(output_text([{'output_type': 'error', 'traceback': ['Traceback', 'ValueError: boom']}]),
        'Traceback\nValueError: boom')
test_eq(output_text([{'output_type': 'display_data', 'data': {'text/markdown': '**hi**', 'text/plain': 'hi'}}]),
        '**hi**')
test_eq(output_text([{'output_type': 'execute_result', 'data': {}}]), '')
test_eq(output_text([{'output_type': 'update_display_data', 'data': {'text/plain': 'x'}}]), '')

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "Result of one execute_request: nbformat-shaped outputs plus the shell reply status."
    ok: bool = True
    execution_count: int | None = None
    outputs: list = field(default_factory=list)
    error: str | None = None
    @property
    def text(self): return output_text(self.outputs)

`ExecOutcome` stores the shell status and nbformat outputs for one execution request. `text` renders the outputs when read. Kernel errors are returned in the outcome.

In [ ]:
out = ExecOutcome(execution_count=3, outputs=[{'output_type': 'stream', 'text': 'fitting\n'}])
out.ok, out.execution_count, out.text

(True, 3, 'fitting\n')

In [ ]:
#| hide
test_eq(ExecOutcome().text, '')
bad = ExecOutcome(ok=False, error='ValueError: boom',
                  outputs=[{'output_type': 'error', 'traceback': ['ValueError: boom']}])
test_eq((bad.ok, bad.text), (False, 'ValueError: boom'))

In [ ]:
#| export
def _runtime_python(python=None):
    "A selected interpreter, or py2app's bundled generic Python helper."
    if python: return str(python)
    if getattr(sys, 'frozen', False):
        helper = Path(sys.executable).with_name('python')
        if helper.exists(): return str(helper)
    return sys.executable

`_runtime_python` returns a selected interpreter. A frozen macOS app uses its bundled Python helper. Other calls use `sys.executable`.

In [ ]:
_runtime_python('/repo/.venv/bin/python'), _runtime_python() == sys.executable

('/repo/.venv/bin/python', True)

In [ ]:
#| export
class KernelStartError(RuntimeError):
    "A kernel that could not start, said in terms of the environment rather than the protocol."

In [ ]:
#| export
def missing_kernel_module(python=None, kernel='ipykernel'):
    "The kernel package `python` cannot import, or None."
    mod = 'ipymini' if kernel == 'ipymini' else 'ipykernel_launcher'
    exe = _runtime_python(python)
    if exe == sys.executable: return None if importlib.util.find_spec(mod) else mod
    src = f"import importlib.util, sys; sys.exit(0 if importlib.util.find_spec({mod!r}) else 1)"
    try: r = subprocess.run([exe, '-c', src], capture_output=True, timeout=20, env=clean_env())
    except (OSError, subprocess.SubprocessError): return None
    return None if r.returncode == 0 else mod

`missing_kernel_module` checks whether an interpreter can import its kernel launcher. It returns `None` when the module exists or the interpreter cannot be checked.

In [ ]:
missing_kernel_module(), missing_kernel_module(kernel='ipymini')

(None, None)

In [ ]:
#| hide
test_eq(missing_kernel_module('/no/such/python'), None)          # cannot tell, so does not say
if Path('/bin/false').exists():
    test_eq(missing_kernel_module('/bin/false'), 'ipykernel_launcher')
    test_eq(missing_kernel_module('/bin/false', 'ipymini'), 'ipymini')

In [ ]:
#| export
def _frozen_pythonpath():
    "Module and extension paths a py2app helper process must inherit."
    if not getattr(sys, 'frozen', False): return None
    resources = Path(sys.executable).resolve().parent.parent/'Resources'
    version = f'python{sys.version_info.major}.{sys.version_info.minor}'
    bundled = [resources/'lib'/version, resources/'lib'/version/'lib-dynload', resources/'lib'/f'python{sys.version_info.major}{sys.version_info.minor}.zip']
    return os.pathsep.join(dict.fromkeys(str(p) for p in [*bundled, *sys.path] if p))

In [ ]:
#| export
def _kernel_env(python=None):
    "Environment for a kernel child, detached from py2app's interpreter redirect."
    env = clean_env()
    if python is None and (path := _frozen_pythonpath()): env['PYTHONPATH'] = path
    return env

`_kernel_env` removes frozen-host interpreter redirection. A bundled interpreter receives the bundle module paths. A selected project interpreter does not.

In [ ]:
#| hide
test_eq(_frozen_pythonpath(), None)                              # this notebook is not a frozen build
test_eq(_kernel_env(), clean_env())
test_eq(_kernel_env('/repo/.venv/bin/python'), clean_env())

In [ ]:
#| export
def _spec(python=None, name='python3', kernel='ipykernel', lang='python', known=None, install=''):
    "Kernelspec pinned to a specific interpreter for Python; other languages return their installed spec unchanged."
    if lang and lang != 'python': return installed_spec(lang, known, install)
    runtime = _runtime_python(python)
    if kernel == 'ipymini': argv = [runtime, '-Xfrozen_modules=off', '-m', 'ipymini', '-f', '{connection_file}']
    else: argv = [runtime, '-m', 'ipykernel_launcher', '-f', '{connection_file}']
    metadata = {'supported_encryption': 'curve'} if kernel == 'ipykernel' else {}
    return KernelSpec(argv=argv, display_name=name, language='python', metadata=metadata)

`_spec` builds a Python kernelspec for a selected interpreter. `ipykernel` enables Curve encryption metadata. Other languages use an installed kernelspec without modification.

In [ ]:
s = _spec('/repo/.venv/bin/python', name='myrepo')
s.argv, s.metadata

(['/repo/.venv/bin/python',
  '-m',
  'ipykernel_launcher',
  '-f',
  '{connection_file}'],
 {'supported_encryption': 'curve'})

In [ ]:
_spec('/repo/.venv/bin/python', kernel='ipymini').argv

['/repo/.venv/bin/python',
 '-Xfrozen_modules=off',
 '-m',
 'ipymini',
 '-f',
 '{connection_file}']

In [ ]:
#| hide
test_eq((s.language, s.display_name), ('python', 'myrepo'))
test_eq(s.argv[-1], '{connection_file}')
test_eq(_spec().argv[0], sys.executable)                         # no interpreter named, this one
mini = _spec('/repo/.venv/bin/python', kernel='ipymini')
test_eq(mini.metadata, {})
assert '-Xfrozen_modules=off' in mini.argv and '-Xfrozen_modules=off' not in s.argv

In [ ]:
#| export
def installed_kernels():
    "Every Jupyter kernelspec on this machine, as `{name: spec}`. An unreadable store is none."
    from jupyter_client.kernelspec import KernelSpecManager
    try: return KernelSpecManager().get_all_specs()
    except Exception: return {}

`installed_kernels` returns all readable Jupyter kernelspecs. An unreadable store returns an empty mapping.

In [ ]:
sorted(installed_kernels())

['ipymini', 'python3', 'python312']

In [ ]:
#| export
def kernelspec_for(lang, known=None):
    "Return the installed kernelspec name for `lang`, checking `known` first; None if not found."
    specs = installed_kernels()
    if (name := (known or {}).get(str(lang))) and name in specs: return name
    return first(n for n, s in specs.items()
                 if str(((s.get('spec') or {}).get('language') or '')).lower() == str(lang).lower())

In [ ]:
#| hide
_real_installed = installed_kernels
def installed_kernels(): return {'python3': {'spec': {'language': 'python'}},
                                 'evcxr': {'spec': {'language': 'Rust'}},
                                 'xrust': {'spec': {'language': 'rust'}}}

`kernelspec_for` finds an installed kernelspec by language. A valid entry in `known` takes priority. Language matching ignores case.

In [ ]:
kernelspec_for('rust'), kernelspec_for('rust', known={'rust': 'xrust'}), kernelspec_for('julia')

('evcxr', 'xrust', None)

In [ ]:
#| hide
test_eq(kernelspec_for('rust', known={'rust': 'not-installed'}), 'evcxr')   # a mapping to nothing is no answer
test_eq(kernelspec_for('python'), 'python3')
installed_kernels = _real_installed

In [ ]:
#| export
def installed_spec(lang, known=None, install=''):
    "The `KernelSpec` for `lang`, or a `KernelStartError` naming what would install one."
    from jupyter_client.kernelspec import KernelSpecManager
    if (name := kernelspec_for(lang, known)): return KernelSpecManager().get_kernel_spec(name)
    how = f' Install one with `{install}`.' if install else ''
    raise KernelStartError(f'no Jupyter kernel is installed for {lang}.{how}')

`installed_spec` returns the kernelspec for a language. A missing kernelspec raises `KernelStartError` with the supplied install command.

In [ ]:
try: installed_spec('rust', install='cargo install evcxr_jupyter')
except KernelStartError as e: print(e)

no Jupyter kernel is installed for rust. Install one with `cargo install evcxr_jupyter`.


In [ ]:
#| hide
test_fail(lambda: _spec(lang='rust'), exc=KernelStartError, contains='no Jupyter kernel is installed for rust')
if kernelspec_for('python'): test_eq(installed_spec('python').language, 'python')
tmp.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()